In [1]:
import pandas as pd
import glob
import os

# 1. 설정: 읽어올 폴더와 저장할 경로 구분
input_folder = r'D:\Project\산업용관리대쉬보드\data\sale'
save_folder = r'D:\Project\산업용관리대쉬보드\data' # <-- 요청하신 저장 경로

output_columns = [
    '매출년월', '용도', '업종분류', '업종', '도로명주소', 
    '지번주소', '고객명', '사용계약번호', '시설물번호', '사용량(m3)', '사용량(mj)'
]

# 2. 모든 CSV 파일 리스트 가져오기
all_files = glob.glob(os.path.join(input_folder, "가정용외_*.csv"))

combined_list = []

for file in all_files:
    try:
        df = pd.read_csv(file, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(file, encoding='utf-8-sig')
    
    # 3. '상품명'이 산업용인 것만 필터링
    if '상품명' in df.columns:
        df = df[df['상품명'].str.contains('산업용', na=False)].copy()
        
        # 4. 숫자 데이터 정리 (콤마 제거 및 숫자형 변환)
        num_cols = ['사용량(m3)', '사용량(mj)']
        for col in num_cols:
            if col in df.columns:
                # 콤마 제거 후 숫자로 변환
                df[col] = df[col].astype(str).str.replace(',', '').replace('nan', '0')
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        # 5. '매출년월' 날짜 타입으로 변환 및 포맷 변경
        if '매출년월' in df.columns:
            df['매출년월'] = pd.to_datetime(df['매출년월'], errors='coerce')

        # 6. 필요한 컬럼만 추출
        available_cols = [col for col in output_columns if col in df.columns]
        combined_list.append(df[available_cols])

# 7. 데이터 최종 병합 및 저장
if combined_list:
    final_df = pd.concat(combined_list, ignore_index=True)
    
    # 매출년월 순으로 정렬
    final_df = final_df.sort_values(by='매출년월')
    
    # 날짜를 yyyy-mm-dd 문자열 형식으로 최종 변환
    final_df['매출년월'] = final_df['매출년월'].dt.strftime('%Y-%m-%d')
    
    # 지정하신 경로에 저장
    save_path = os.path.join(save_folder, '산업용_통합데이터_정리완료.csv')
    final_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ 작업 완료! 파일이 다음 경로에 저장되었습니다: {save_path}")
    print(f"📊 총 {len(final_df):,}건의 산업용 데이터를 통합했습니다.")
else:
    print("❌ 조건에 맞는 데이터가 없습니다. 파일명이나 '상품명' 컬럼을 확인해주세요.")

✅ 작업 완료! 파일이 다음 경로에 저장되었습니다: D:\Project\산업용관리대쉬보드\data\산업용_통합데이터_정리완료.csv
📊 총 179,155건의 산업용 데이터를 통합했습니다.
